## Extracción de Datos

In [2]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Configurar cliente con caché y reintentos
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 7.9,
    "longitude": -72.5,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
    ],
    "timezone": "America/Bogota",
    "past_days": 5,
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

print(f"Coordenadas: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevación: {response.Elevation()} m snm")
print(f"Offset UTC: {response.UtcOffsetSeconds()}s")

# Procesar variables horarias — el índice debe coincidir con el orden en `hourly`
hourly = response.Hourly()
hourly_temperature_2m      = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity   = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation       = hourly.Variables(2).ValuesAsNumpy()
hourly_rain                = hourly.Variables(3).ValuesAsNumpy()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    ),
    "temperature_2m": hourly_temperature_2m,
    "relative_humidity_2m": hourly_relative_humidity,
    "precipitation": hourly_precipitation,
    "rain": hourly_rain
}

df = pd.DataFrame(data=hourly_data)
print("\nDatos horarios\n", df)

Coordenadas: 7.90861177444458°N -72.49148559570312°E
Elevación: 299.0 m snm
Offset UTC: -18000s

Datos horarios
                          date  temperature_2m  relative_humidity_2m  \
0   2026-04-29 05:00:00+00:00       23.650000             84.330582   
1   2026-04-29 06:00:00+00:00       23.750000             91.043976   
2   2026-04-29 07:00:00+00:00       23.900000             92.447327   
3   2026-04-29 08:00:00+00:00       23.350000             95.561501   
4   2026-04-29 09:00:00+00:00       23.200001             95.556602   
..                        ...             ...                   ...   
283 2026-05-11 00:00:00+00:00       28.850000             54.453133   
284 2026-05-11 01:00:00+00:00       28.049999             57.223133   
285 2026-05-11 02:00:00+00:00       27.299999             59.600933   
286 2026-05-11 03:00:00+00:00       26.600000             61.909901   
287 2026-05-11 04:00:00+00:00       25.950001             63.731808   

     precipitation  rain  
0      

In [3]:
df_bogota=df.copy()
df_bogota["date"] = df_bogota["date"].dt.tz_convert("America/Bogota")
df_bogota

,date,temperature_2m,relative_humidity_2m,precipitation,rain
0,2026-04-29 00:00:00-05:00,23.650000,84.330582,0.0,0.0
1,2026-04-29 01:00:00-05:00,23.750000,91.043976,0.0,0.0
2,2026-04-29 02:00:00-05:00,23.900000,92.447327,0.0,0.0
3,2026-04-29 03:00:00-05:00,23.350000,95.561501,0.0,0.0
4,2026-04-29 04:00:00-05:00,23.200001,95.556602,0.0,0.0
...,...,...,...,...,...
283,2026-05-10 19:00:00-05:00,28.850000,54.453133,0.0,0.0
284,2026-05-10 20:00:00-05:00,28.049999,57.223133,0.0,0.0
285,2026-05-10 21:00:00-05:00,27.299999,59.600933,0.0,0.0
286,2026-05-10 22:00:00-05:00,26.600000,61.909901,0.0,0.0


In [4]:
df_bogota["time"]  = df_bogota["date"].dt.time
df_bogota["date"] = df_bogota["date"].dt.date
df_bogota=df_bogota[["date","time","temperature_2m","relative_humidity_2m","precipitation","rain"]]
df_bogota

,date,time,temperature_2m,relative_humidity_2m,precipitation,rain
0,2026-04-29,00:00:00,23.650000,84.330582,0.0,0.0
1,2026-04-29,01:00:00,23.750000,91.043976,0.0,0.0
2,2026-04-29,02:00:00,23.900000,92.447327,0.0,0.0
3,2026-04-29,03:00:00,23.350000,95.561501,0.0,0.0
4,2026-04-29,04:00:00,23.200001,95.556602,0.0,0.0
...,...,...,...,...,...,...
283,2026-05-10,19:00:00,28.850000,54.453133,0.0,0.0
284,2026-05-10,20:00:00,28.049999,57.223133,0.0,0.0
285,2026-05-10,21:00:00,27.299999,59.600933,0.0,0.0
286,2026-05-10,22:00:00,26.600000,61.909901,0.0,0.0


### Verificación de los datos

In [5]:
# hay datos nulos
df_bogota.isnull().sum()

date                    0
time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
rain                    0
dtype: int64

## Análisis básico

In [6]:
# ¿Cuál es la temperatura promedio?
temperatura_promedio=df_bogota["temperature_2m"].mean()
display(temperatura_promedio)

np.float32(27.297743)

In [9]:
# ¿Cuándo llueve más?
# ¿a qué hora llueve más en promedio?
lluvia_por_hora = df_bogota.groupby("time")["rain"].mean().sort_values()
display(lluvia_por_hora.head(5))
# ¿qué día llovió más?
lluvia_por_dia = df_bogota.groupby("date")["rain"].sum().sort_values()
display(lluvia_por_dia.head(5))

# El momento exacto de mayor lluvia
idx = df_bogota["rain"].idxmax()
print(df_bogota.loc[idx, ["date", "time", "rain"]])


time
00:00:00    0.0
01:00:00    0.0
02:00:00    0.0
05:00:00    0.0
07:00:00    0.0
Name: rain, dtype: float32

date
2026-04-30    0.0
2026-05-01    0.0
2026-05-05    0.0
2026-05-03    0.0
2026-05-06    0.0
Name: rain, dtype: float32

date    2026-05-02
time      14:00:00
rain           0.3
Name: 86, dtype: object
